# Mirrored Entity Resolution & AML Pipeline
This notebook demonstrates the new "Mirrored" architecture.

### Pipeline Stages:
1. **Feature Engineering:** Calculate Peak Vectors & Shift Scores.
2. **Resolution (The Fork):** Split Regular vs Super Nodes. Cluster Super Nodes.
3. **Anomaly Detection:** Identify High-Risk Entities.
4. **Mirrored Linking:** Connect Source/Sink pairs (Syndicates).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.gnn.config.schema import Schema
from src.gnn.entity_resolution.features import AdvancedFeatureEngineer
from src.gnn.entity_resolution.resolution import EntityResolver
from src.gnn.entity_resolution.anomaly import AnomalyDetector
from src.gnn.entity_resolution.linking import SyndicateLinker

print("Modules loaded.")

## 1. Synthetic Data Generation

In [ ]:
def generate_data():
    data = []
    
    # 1. Regular Users
    for i in range(50):
        data.append({
            Schema.CUSTOMER_NAME: f"User_{i}",
            Schema.BANK_NAME: "Bank_A",
            Schema.BANK_ACCOUNT: f"Acc_User_{i}",
            Schema.VOLUME: np.random.uniform(100, 500),
            Schema.DIRECTION: np.random.choice(['Inbound', 'Outbound']),
            Schema.DATE: pd.Timestamp("2024-01-01") + pd.Timedelta(hours=i)
        })
        
    # 2. Super Node "John Smith" (Mixed Bag)
    # Cluster A: The "Real" John Smith (High Volume, Round Amounts)
    for i in range(5):
        data.append({
            Schema.CUSTOMER_NAME: "John Smith",
            Schema.BANK_NAME: "Bank_B",
            Schema.BANK_ACCOUNT: f"JS_Real_{i}",
            Schema.VOLUME: 5000.0, # Round
            Schema.DIRECTION: 'Outbound',
            Schema.DATE: pd.Timestamp("2024-01-02 10:00:00")
        })
    # Cluster B: Random Noise John Smiths
    for i in range(10):
         data.append({
            Schema.CUSTOMER_NAME: "John Smith",
            Schema.BANK_NAME: "Bank_C",
            Schema.BANK_ACCOUNT: f"JS_Noise_{i}",
            Schema.VOLUME: np.random.uniform(50, 150), # Random
            Schema.DIRECTION: 'Inbound',
            Schema.DATE: pd.Timestamp("2024-01-03") + pd.Timedelta(minutes=i*10)
        })
        
    # 3. Mirrored Pair (The Syndicate)
    # Mule A (Source): Bursts of large Outbound
    for i in range(5):
        data.append({
            Schema.CUSTOMER_NAME: "Mule_Source",
            Schema.BANK_NAME: "Bank_D",
            Schema.BANK_ACCOUNT: "Mule_Acc_A",
            Schema.VOLUME: 9900.0,
            Schema.DIRECTION: 'Outbound',
            Schema.DATE: pd.Timestamp("2024-01-10 12:00:00")
        })
        
    # Mule B (Sink): Bursts of large Inbound (Matching A)
    for i in range(5):
        data.append({
            Schema.CUSTOMER_NAME: "Mule_Sink",
            Schema.BANK_NAME: "Bank_E",
            Schema.BANK_ACCOUNT: "Mule_Acc_B",
            Schema.VOLUME: 9900.0,
            Schema.DIRECTION: 'Inbound',
            Schema.DATE: pd.Timestamp("2024-01-10 13:00:00")
        })

    return pd.DataFrame(data)

raw_df = generate_data()
print(f"Generated {len(raw_df)} transactions.")

## 2. Advanced Feature Engineering (Peak & Shift)

In [ ]:
engineer = AdvancedFeatureEngineer()
feature_df = engineer.fit_transform(raw_df)

print("Feature DataFrame (One row per Account):")
print(feature_df[[Schema.CUSTOMER_NAME, Schema.PEAK_FLOW_THROUGH, Schema.PEAK_ROUNDNESS]].head())

## 3. Phase 1: Resolution (Super Node Splitting)

In [ ]:
# Threshold = 20 accounts. 
# Our John Smith has 25 accounts (10 Real + 15 Noise), so he is a Super Node.
resolver = EntityResolver(supernode_threshold=20)
resolved_df = resolver.resolve(feature_df)

print("Super Node Resolution Results for John Smith:")
js_df = resolved_df[resolved_df[Schema.CUSTOMER_NAME] == "John Smith"]
print(js_df[[Schema.BANK_ACCOUNT, Schema.RESOLVED_ENTITY_ID, Schema.PEAK_ROUNDNESS]].head(25))
print("\nUnique Entities for John Smith:", js_df[Schema.RESOLVED_ENTITY_ID].nunique())

## 4. Phase 2: Anomaly Detection

In [ ]:
detector = AnomalyDetector(contamination=0.2)
risk_df = detector.detect(resolved_df)

print("Top High Risk Entities:")
high_risk = risk_df[risk_df['is_high_risk']].sort_values(Schema.RISK_SCORE, ascending=False)
print(high_risk)

## 5. Phase 3: Mirrored Linking (Syndicate Detection)

In [ ]:
linker = SyndicateLinker()
links = linker.link(risk_df, raw_df, resolved_df)

print("Detected Syndicates:")
print(links)